# 213. HyDE：怎样用假想文档提升召回但不把幻觉送进答案？

> **面试问题：怎样分开 query/HyDE 双路召回、用 RRF 融合、只组装真实文档，并评测检索增益、grounding 与生成成本？**

## 先给结论

不要只背论文名或框架名。应当说明输入/状态合同、核心算法、失败分支、独立 oracle、指标和可回滚制品。以下均使用受控小数据验证实现机制；真实生产仍需替换模型、权限、索引、安全审计与线上评测。

## 一手资料

- [HyDE](https://arxiv.org/abs/2212.10496)
- [RAG](https://arxiv.org/abs/2005.11401)
- [RRF](https://dl.acm.org/doi/10.1145/1571941.1572114)

In [ ]:
contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "versioned"}  # 执行本行的状态、计算或校验逻辑。
assert contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert contract["production"] == "versioned"  # 执行本行的状态、计算或校验逻辑。
assert len(contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 真实语料与假想文档边界

HyDE 的假想文档用于构造更接近相关文档的检索向量，它不是证据也不能直接回答用户。真实文档、ACL、版本和引用仍是 grounding 的唯一来源。


In [ ]:
documents = {"d1": "退款需要人工确认后处理", "d2": "配送通常三天完成", "d3": "退款原路返回付款账户"}  # 执行本行的状态、计算或校验逻辑。
query = "退款如何处理"  # 执行本行的状态、计算或校验逻辑。
hypothesis = "退款需要确认并返回付款账户"  # 执行本行的状态、计算或校验逻辑。
assert "退款" in query  # 执行本行的状态、计算或校验逻辑。
assert "确认" in hypothesis  # 执行本行的状态、计算或校验逻辑。
assert len(documents) == 3  # 执行本行的状态、计算或校验逻辑。


## 2. 统一 encoder 接口

真实 HyDE 使用 embedding 模型，教学用字符集合相似度暴露同一接口。query/hypothesis/document 必须经同一 encoder 版本处理，且 hypothesis 不得被写回文档索引。


In [ ]:
def terms(text):  # 执行本行的状态、计算或校验逻辑。
    return set(text)  # 执行本行的状态、计算或校验逻辑。
def similarity(left, right):  # 执行本行的状态、计算或校验逻辑。
    return len(terms(left) & terms(right)) / max(1, len(terms(left) | terms(right)))  # 执行本行的状态、计算或校验逻辑。
assert similarity("退款", "退款确认") > 0  # 执行本行的状态、计算或校验逻辑。
assert similarity("退款", "配送") == 0.0  # 执行本行的状态、计算或校验逻辑。
assert similarity(query, hypothesis) > 0  # 执行本行的状态、计算或校验逻辑。


## 3. 双路候选

保留 query-only 和 HyDE 两路结果有利于诊断。HyDE 可能补充答案语气，也可能引入偏题实体；每路 rank/score 都需要记录，不能只保留融合后的黑盒列表。


In [ ]:
def rank(text):  # 执行本行的状态、计算或校验逻辑。
    return [doc_id for doc_id, _ in sorted(documents.items(), key=lambda item: similarity(text, item[1]), reverse=True) if similarity(text, documents[doc_id]) > 0]  # 执行本行的状态、计算或校验逻辑。
query_rank = rank(query)  # 执行本行的状态、计算或校验逻辑。
hyde_rank = rank(hypothesis)  # 执行本行的状态、计算或校验逻辑。
assert query_rank[0] == "d1"  # 执行本行的状态、计算或校验逻辑。
assert "d3" in hyde_rank  # 执行本行的状态、计算或校验逻辑。
assert all(doc_id in documents for doc_id in hyde_rank)  # 执行本行的状态、计算或校验逻辑。


## 4. RRF 融合

不同检索器分数通常不可直接相加。RRF 将 rank 转为统一的倒数贡献，能融合 query/hypothesis 路；k、权重、候选数应在验证集选择并保留每路贡献。


In [ ]:
def rrf(rankings, k=60):  # 执行本行的状态、计算或校验逻辑。
    scores = {}  # 执行本行的状态、计算或校验逻辑。
    for ranking in rankings:  # 执行本行的状态、计算或校验逻辑。
        for index, doc_id in enumerate(ranking, 1):  # 执行本行的状态、计算或校验逻辑。
            scores[doc_id] = scores.get(doc_id, 0.0) + 1 / (k + index)  # 执行本行的状态、计算或校验逻辑。
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda item: item[1], reverse=True)]  # 执行本行的状态、计算或校验逻辑。
fused = rrf([query_rank, hyde_rank])  # 执行本行的状态、计算或校验逻辑。
assert fused[0] == "d1"  # 执行本行的状态、计算或校验逻辑。
assert set(fused).issubset(documents)  # 执行本行的状态、计算或校验逻辑。
assert len(fused) >= 2  # 执行本行的状态、计算或校验逻辑。


## 5. grounded context

组装上下文时只选择 fused 后的真实文档，并带 doc id/版本用于引用。假想文档的错误细节不能进入 answer prompt；无真实证据时应拒答、澄清或继续检索。


In [ ]:
def assemble(doc_ids, budget):  # 执行本行的状态、计算或校验逻辑。
    return [{"doc_id": doc_id, "text": documents[doc_id]} for doc_id in doc_ids[:budget]]  # 执行本行的状态、计算或校验逻辑。
context = assemble(fused, 2)  # 执行本行的状态、计算或校验逻辑。
assert all(item["doc_id"] in documents for item in context)  # 执行本行的状态、计算或校验逻辑。
assert all(item["text"] != hypothesis for item in context)  # 执行本行的状态、计算或校验逻辑。
assert context[0]["doc_id"] == "d1"  # 执行本行的状态、计算或校验逻辑。


## 6. 偏题 hypothesis 防护

生成器可能产生与问题无关的假想文本。可用 query overlap、领域/ACL filter、双路融合和 reranker 门控；完全无交集时应该降级 query-only 或请求澄清，而非盲信假想内容。


In [ ]:
bad_rank = rank("配送三天完成")  # 执行本行的状态、计算或校验逻辑。
def overlap_guard(left, right):  # 执行本行的状态、计算或校验逻辑。
    return bool(set(left) & set(right))  # 执行本行的状态、计算或校验逻辑。
assert bad_rank[0] == "d2"  # 执行本行的状态、计算或校验逻辑。
assert overlap_guard(query_rank, hyde_rank)  # 执行本行的状态、计算或校验逻辑。
assert not overlap_guard(["d1"], ["d2"])  # 执行本行的状态、计算或校验逻辑。


## 7. 检索与回答评测

HyDE 可能提升 Recall@k 却增加生成/编码成本。评测需对比 query-only、hyde-only、fusion，并额外验证最终回答的 claim 是否来自真实 context，而非 hypothesis。


In [ ]:
def recall_at_k(ranking, gold, k):  # 执行本行的状态、计算或校验逻辑。
    return int(gold in ranking[:k])  # 执行本行的状态、计算或校验逻辑。
metrics = {"query": recall_at_k(query_rank, "d3", 1), "hyde": recall_at_k(hyde_rank, "d3", 2), "fusion": recall_at_k(fused, "d3", 2)}  # 执行本行的状态、计算或校验逻辑。
assert metrics["query"] == 0  # 执行本行的状态、计算或校验逻辑。
assert metrics["hyde"] == 1  # 执行本行的状态、计算或校验逻辑。
assert metrics["fusion"] == 1  # 执行本行的状态、计算或校验逻辑。


## 8. 链路制品

HyDE 的结果由生成器提示/模型、encoder、fusion 参数、文档 index、ACL 和 context budget 共同决定。它们必须同 trace 保存，才能解释检索漂移或从线上问题复放。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"generator": "hyde-v1", "encoder": "set-v1", "fusion": "rrf-60", "index": "docs-v1", "budget": 2}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["budget"] == len(context)  # 执行本行的状态、计算或校验逻辑。
assert artifact["fusion"] == "rrf-60"  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整答案应先定义成功条件，再说明数据状态、主路径、失败边界和评测。受控断言只证明实现不变量，不能直接外推为真实大语料、模型语义、线上成本或安全效果。
